In [ ]:
import numpy as np
import xgboost as xgb
import optuna
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# input dataset
data_test = pd.read_csv('/content/drive/MyDrive/DSA/CS3244/data/hapt_test.csv')
data_train = pd.read_csv('/content/drive/MyDrive/DSA/CS3244/data/hapt_train.csv')

In [4]:
# seperate into x and y and modify to zero indexing for pca
data_test['activity_id'] = data_test['activity_id']-1
test_x = data_test.drop(columns=['subject_id', 'activity_label', 'activity_id'])
test_y = data_test['activity_id'].astype('category')

data_train['activity_id'] = data_train['activity_id']-1
train_x = data_train.drop(columns=['subject_id', 'activity_label', 'activity_id'])
train_y = data_train['activity_id'].astype('category')

# Stock multinomial logistic regression

In [5]:
X = train_x
y = train_y

# Create pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500))
])

# Define grid of parameters to evaluate
param_grid = {
    'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]
}

# Perform 5-fold cross-validation grid search
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X, y)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best 5-Fold Cross-Validation Score: {grid.best_score_:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best Parameters: {'clf__C': 1.0}
Best 5-Fold Cross-Validation Score: 0.9246


In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(test_x)
model_logreg = LogisticRegression(
    multi_class='multinomial', solver='lbfgs', max_iter=500, C=1
)
model_logreg.fit(X_scaled, y)

y_pred = model_logreg.predict(X_test_scaled)
y_prob = model_logreg.predict_proba(X_scaled)

print(f'Accuracy: {accuracy_score(test_y, y_pred):.4f}\n')
# Convert unique targets to sorted strings to prevent TypeError
target_names = [str(label) for label in sorted(test_y.unique())]
print(classification_report(test_y, y_pred, target_names=target_names))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.9421

              precision    recall  f1-score   support

           0       0.94      0.99      0.97       496
           1       0.96      0.94      0.95       471
           2       0.99      0.95      0.97       420
           3       0.97      0.87      0.92       508
           4       0.89      0.98      0.93       556
           5       1.00      0.99      1.00       545
           6       0.75      0.78      0.77        23
           7       1.00      1.00      1.00        10
           8       0.60      0.56      0.58        32
           9       0.72      0.72      0.72        25
          10       0.64      0.71      0.67        49
          11       0.71      0.63      0.67        27

    accuracy                           0.94      3162
   macro avg       0.85      0.84      0.85      3162
weighted avg       0.94      0.94      0.94      3162



# PCA dimension reduction

In [7]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

# ==============================================================================
# Step 1: Load Data
# ==============================================================================
X_train = train_x
y_train = train_y
subject_train = data_train['subject_id']

X_test = test_x
y_test = test_y
subject_test = data_test['subject_id']

# Helper function for Step 3: Undersample basic activities (0-5)
def undersample_fold(X, y, groups, seed):
    rng = np.random.default_rng(seed)
    selected_indices = []

    for label in np.unique(y):
        label_indices = np.where(y == label)[0]
        if label <= 5:  # Basic activities: retain at most 300 windows
            if len(label_indices) > 300:
                chosen = rng.choice(label_indices, size=300, replace=False)
            else:
                chosen = label_indices
        else:  # Transition windows: keep all
            chosen = label_indices
        selected_indices.extend(chosen)

    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)  # Shuffle order of combined indices
    return X.iloc[selected_indices] if hasattr(X, 'iloc') else X[selected_indices], y.iloc[selected_indices] if hasattr(y, 'iloc') else y[selected_indices], groups.iloc[selected_indices] if hasattr(groups, 'iloc') else groups[selected_indices]


# ==============================================================================
# Steps 2–6: Stratified Group K-Fold Cross-Validation & Tuning
# ==============================================================================
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

val_scores = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=subject_train)):
    # 2. Split into train/validation folds using .iloc for safe pandas indexing
    X_tr, y_tr, sub_tr = X_train.iloc[train_idx], y_train.iloc[train_idx], subject_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    # 3. Randomly retain at most 300 windows per basic activity (0-5) inside training fold
    seed = 42 + fold
    X_tr_sub, y_tr_sub, _ = undersample_fold(X_tr, y_tr, sub_tr, seed=seed)

    # 4. Fit StandardScaler on undersampled training fold only
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_sub)
    X_val_scaled = scaler.transform(X_val)

    # 5. Fit Randomized PCA (200 components) on undersampled training fold only
    pca = PCA(n_components=200, svd_solver="randomized", random_state=seed)
    X_tr_pca = pca.fit_transform(X_tr_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    # 6. Compute class weights on undersampled training labels & train model
    classes = np.unique(y_tr_sub)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr_sub)
    class_weight_dict = dict(zip(classes, weights))

    clf = RandomForestClassifier(random_state=42, class_weight=class_weight_dict)
    clf.fit(X_tr_pca, y_tr_sub)

    # Evaluate on untouched validation set
    val_score = clf.score(X_val_pca, y_val)
    val_scores.append(val_score)
    print(f"Fold {fold + 1} Validation Accuracy: {val_score:.4f}")

print(f"Mean Validation Accuracy: {np.mean(val_scores):.4f}")


# ==============================================================================
# Step 7: Final Training on All 21 Training Participants & Testing
# ==============================================================================
# Repeat undersampling on full training dataset using seed 42
X_train_sub, y_train_sub, _ = undersample_fold(X_train, y_train, subject_train, seed=42)

# Fit scaler on undersampled full training set
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_sub)
X_test_scaled = final_scaler.transform(X_test)

# Fit randomized PCA on undersampled full training set
final_pca = PCA(n_components=200, svd_solver="randomized", random_state=42)
X_train_pca = final_pca.fit_transform(X_train_scaled)
X_test_pca = final_pca.transform(X_test_scaled)

# Compute class weights on full undersampled training labels & fit final model
final_classes = np.unique(y_train_sub)
final_weights = compute_class_weight(class_weight="balanced", classes=final_classes, y=y_train_sub)
final_class_weight_dict = dict(zip(final_classes, final_weights))




Fold 1 Validation Accuracy: 0.8776
Fold 2 Validation Accuracy: 0.8250
Fold 3 Validation Accuracy: 0.8120
Mean Validation Accuracy: 0.8382


# Random forest with PCA

In [8]:
RandomForest_model = RandomForestClassifier(random_state=42, class_weight=final_class_weight_dict)
RandomForest_model.fit(X_train_pca, y_train_sub)

# Evaluate final test accuracy on untouched test windows
test_accuracy = RandomForest_model.score(X_test_pca, y_test)
print(f"Final Test Accuracy: {test_accuracy:.4f}")

y_pred = RandomForest_model.predict(X_test_pca)
y_prob = RandomForest_model.predict_proba(X_train_pca)

print(f'Accuracy: {accuracy_score(test_y, y_pred):.4f}\n')
# Convert unique targets to sorted strings to prevent TypeError
target_names = [str(label) for label in sorted(test_y.unique())]
print(classification_report(test_y, y_pred, target_names=target_names))

Final Test Accuracy: 0.8583
Accuracy: 0.8583

              precision    recall  f1-score   support

           0       0.83      0.93      0.87       496
           1       0.85      0.87      0.86       471
           2       0.86      0.78      0.82       420
           3       0.88      0.75      0.81       508
           4       0.80      0.92      0.86       556
           5       0.99      0.95      0.97       545
           6       0.94      0.65      0.77        23
           7       1.00      0.80      0.89        10
           8       0.68      0.66      0.67        32
           9       0.66      0.76      0.70        25
          10       0.67      0.59      0.63        49
          11       0.76      0.48      0.59        27

    accuracy                           0.86      3162
   macro avg       0.83      0.76      0.79      3162
weighted avg       0.86      0.86      0.86      3162



# Gradient boosting via XGBoost with PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# Optuna logging verbosity
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# Helper Function: Undersample fold data (Step 3)
# ==============================================================================
def undersample_fold(X, y, groups, seed):
    rng = np.random.default_rng(seed)
    selected_indices = []

    for label in np.unique(y):
        label_indices = np.where(y == label)[0]
        if label <= 5:  # Basic activities: retain at most 300 windows
            if len(label_indices) > 300:
                chosen = rng.choice(label_indices, size=300, replace=False)
            else:
                chosen = label_indices
        else:  # Transition windows: keep all
            chosen = label_indices
        selected_indices.extend(chosen)

    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)
    return X.iloc[selected_indices] if hasattr(X, 'iloc') else X[selected_indices], y.iloc[selected_indices] if hasattr(y, 'iloc') else y[selected_indices], groups.iloc[selected_indices] if hasattr(groups, 'iloc') else groups[selected_indices]


# ==============================================================================
# Objective Function for Optuna
# ==============================================================================
def objective(trial):
    # Hyperparameter search space
    params = {
        "objective": "multi:softprob",
        "num_class": 12,
        "eval_metric": "mlogloss",
        "random_state": 42,
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }

    sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    val_scores = []

    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=subject_train)):
        # 1. Split train/validation using .iloc for safe pandas indexing
        X_tr, y_tr, sub_tr = X_train.iloc[train_idx], y_train.iloc[train_idx], subject_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        # 2. Undersample training fold only
        seed = 42 + fold
        X_tr_sub, y_tr_sub, _ = undersample_fold(X_tr, y_tr, sub_tr, seed=seed)

        # 3. Fit scaler on training fold only
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr_sub)
        X_val_scaled = scaler.transform(X_val)

        # 4. Fit PCA on training fold only
        pca = PCA(n_components=200, svd_solver="randomized", random_state=seed)
        X_tr_pca = pca.fit_transform(X_tr_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        # 5. Compute sample weights on undersampled training labels
        sample_weights_tr = compute_sample_weight("balanced", y_tr_sub)

        # 6. Fit XGBoost
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr_pca, y_tr_sub, sample_weight=sample_weights_tr)

        # Evaluate on validation fold
        val_acc = clf.score(X_val_pca, y_val)
        val_scores.append(val_acc)

    return np.mean(val_scores)


# ==============================================================================
# Run Tuning & Train Final Model
# ==============================================================================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # Adjust n_trials as needed

print(f"Best Validation Accuracy: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------------------------
# Final Pipeline Run on Full Training Data (Step 7)
# ------------------------------------------------------------------------------
best_params = study.best_params.copy()
best_params.update({
    "objective": "multi:softprob",
    "num_class": 12,
    "eval_metric": "mlogloss",
    "random_state": 42
})

# 1. Undersample full train set using seed 42
X_train_sub, y_train_sub, _ = undersample_fold(train_x, train_y, subject_train, seed=42)

# 2. Fit final scaler & transform test set
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_sub)
X_test_scaled = final_scaler.transform(test_x)

# 3. Fit final PCA & transform test set
final_pca = PCA(n_components=200, svd_solver="randomized", random_state=42)
X_train_pca = final_pca.fit_transform(X_train_scaled)
X_test_pca = final_pca.transform(X_test_scaled)

# 4. Compute sample weights on final undersampled labels
sample_weights_final = compute_sample_weight("balanced", y_train_sub)

# 5. Train tuned XGBoost model & evaluate
XGB_model = xgb.XGBClassifier(**best_params)
XGB_model.fit(X_train_pca, y_train_sub, sample_weight=sample_weights_final)

final_test_acc = XGB_model.score(X_test_pca, test_y)
print(f"\nFinal Test Accuracy with Tuned XGBoost: {final_test_acc:.4f}")

Best Validation Accuracy: 0.8719
Best Hyperparameters:
  n_estimators: 500
  max_depth: 6
  learning_rate: 0.1378353795412322
  subsample: 0.6160797325433003
  colsample_bytree: 0.8472596494285396
  gamma: 0.0007492283178950921
  min_child_weight: 7

Final Test Accuracy with Tuned XGBoost: 0.9035


In [18]:
y_pred = XGB_model.predict(X_test_pca)
y_prob = XGB_model.predict_proba(X_train_pca)

print(f'Accuracy: {accuracy_score(test_y, y_pred):.4f}\n')
# Convert unique targets to sorted strings to prevent TypeError
target_names = [str(label) for label in sorted(test_y.unique())]
print(classification_report(test_y, y_pred, target_names=target_names))

Accuracy: 0.9035

              precision    recall  f1-score   support

           0       0.87      0.96      0.91       496
           1       0.93      0.93      0.93       471
           2       0.93      0.82      0.87       420
           3       0.92      0.83      0.88       508
           4       0.86      0.93      0.90       556
           5       0.99      0.98      0.99       545
           6       0.58      0.65      0.61        23
           7       0.83      1.00      0.91        10
           8       0.67      0.81      0.73        32
           9       0.67      0.56      0.61        25
          10       0.79      0.76      0.77        49
          11       0.61      0.63      0.62        27

    accuracy                           0.90      3162
   macro avg       0.80      0.82      0.81      3162
weighted avg       0.91      0.90      0.90      3162



# Linear SVM with PCA

In [13]:
from sklearn.svm import LinearSVC

# ==============================================================================
# Inside Fold Loop (Step 6):
# ==============================================================================
# LinearSVC automatically computes class weights from training fold labels
clf = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=2000,
    dual="auto"  # 'auto' selects primal or dual optimization based on sample/feature count
)
clf.fit(X_tr_pca, y_tr_sub)

val_score = clf.score(X_val_pca, y_val)


# ==============================================================================
# Final Model Training (Step 7):
# ==============================================================================
LinearSVM_model = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=2000,
    dual="auto"
)
LinearSVM_model.fit(X_train_pca, y_train_sub)

test_accuracy = LinearSVM_model.score(X_test_pca, y_test)
print(f"Final Linear SVM Test Accuracy: {test_accuracy:.4f}")

y_pred = LinearSVM_model.predict(X_test_pca)
# Use decision_function instead of predict_proba
y_score = LinearSVM_model.decision_function(X_train_pca)

print(f'Accuracy: {accuracy_score(test_y, y_pred):.4f}\n')
# Convert unique targets to sorted strings to prevent TypeError
target_names = [str(label) for label in sorted(test_y.unique())]
print(classification_report(test_y, y_pred, target_names=target_names))

Final Linear SVM Test Accuracy: 0.8887
Accuracy: 0.8887

              precision    recall  f1-score   support

           0       0.93      0.95      0.94       496
           1       0.97      0.86      0.91       471
           2       0.99      0.91      0.95       420
           3       0.85      0.81      0.83       508
           4       0.81      0.90      0.85       556
           5       1.00      0.96      0.98       545
           6       0.57      0.74      0.64        23
           7       0.71      1.00      0.83        10
           8       0.51      0.66      0.58        32
           9       0.52      0.68      0.59        25
          10       0.44      0.65      0.52        49
          11       0.52      0.59      0.55        27

    accuracy                           0.89      3162
   macro avg       0.73      0.81      0.76      3162
weighted avg       0.90      0.89      0.89      3162

